# 04 — Измерение производительности методов интерполяции

Измерение времени работы пяти методов интерполяции для дополнения таблицы 3 ВКР колонкой «Время, мкс/точка». Бенчмарк проводится на одном представительном треке (~3000 точек) при шаге прореживания 5 с 1000 повторениями для статистической устойчивости.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd

from interp_research.plotting import set_thesis_style
from interp_research.methods import linear, lagrange, cubic_spline, b_spline, motion_aware
from interp_research.holdout import holdout_split_2d
from interp_research.benchmark import benchmark_method

set_thesis_style()

In [2]:
PROCESSED_DIR = Path("../data/processed")
TABLES_DIR = Path("../results/tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

TRACK_FILE = "ottawa-2016-06-05__0.npz"
STEP = 5
N_RUNS = 1000
N_WARMUP = 10
N_QUERY = 100

METHODS = {
    "Линейная": linear.interpolate,
    "Лагранж": lambda t_k, x_k, t_q: lagrange.interpolate(t_k, x_k, t_q, degree=8),
    "Кубический сплайн": cubic_spline.interpolate,
    "B-сплайн": b_spline.interpolate,
    "Motion-aware": motion_aware.interpolate,
}

## Подготовка данных

Загружаем трек ottawa-2016-06-05 (2931 точка), разбиваем с шагом 5. Из контрольных точек берём случайную подвыборку из 100 штук — это `t_query`, на которых будем замерять время.

In [3]:
data = np.load(PROCESSED_DIR / TRACK_FILE)
t_full = data["t"]
t_full = t_full - t_full[0]
x_full = data["x_local"]
y_full = data["y_local"]

t_known, x_known, y_known, t_held, x_held, y_held = holdout_split_2d(
    t_full, x_full, y_full, step=STEP
)

rng = np.random.default_rng(42)
query_idx = np.sort(rng.choice(len(t_held), size=N_QUERY, replace=False))
t_query = t_held[query_idx]

print(f"Трек: {len(t_full)} точек")
print(f"Узлы интерполяции: {len(t_known)}")
print(f"Точки запроса (t_query): {len(t_query)}")

Трек: 2931 точек
Узлы интерполяции: 587
Точки запроса (t_query): 100


## Бенчмарк всех методов

Для каждого метода замеряем время отдельно по координатам X и Y, затем усредняем. Время нормируем на количество точек запроса → мкс/точка.

In [4]:
bench_rows: list[dict] = []

for method_name, method_fn in METHODS.items():
    stats_x = benchmark_method(
        method_fn, t_known, x_known, t_query,
        n_runs=N_RUNS, n_warmup=N_WARMUP,
    )
    stats_y = benchmark_method(
        method_fn, t_known, y_known, t_query,
        n_runs=N_RUNS, n_warmup=N_WARMUP,
    )
    mean_us = (stats_x["mean_us"] + stats_y["mean_us"]) / 2
    median_us = (stats_x["median_us"] + stats_y["median_us"]) / 2
    std_us = (stats_x["std_us"] + stats_y["std_us"]) / 2

    bench_rows.append({
        "Метод": method_name,
        "Среднее, мкс": mean_us,
        "Медиана, мкс": median_us,
        "Std, мкс": std_us,
        "мкс/точка": mean_us / N_QUERY,
    })
    print(f"{method_name:>20s}: {mean_us:8.0f} мкс  ({mean_us / N_QUERY:.1f} мкс/точка)")

df_bench = pd.DataFrame(bench_rows)
df_bench

            Линейная:       23 мкс  (0.2 мкс/точка)


             Лагранж:     5896 мкс  (59.0 мкс/точка)


   Кубический сплайн:     2891 мкс  (28.9 мкс/точка)


            B-сплайн:      265 мкс  (2.7 мкс/точка)


        Motion-aware:      153 мкс  (1.5 мкс/точка)


,Метод,"Среднее, мкс","Медиана, мкс","Std, мкс",мкс/точка
0,Линейная,22.964115,19.790498,11.920146,0.229641
1,Лагранж,5896.215981,5622.933000,1131.147387,58.962160
2,Кубический сплайн,2890.848349,2702.892251,673.188422,28.908483
3,B-сплайн,265.010706,224.199501,206.749308,2.650107
4,Motion-aware,153.382724,129.893249,131.400659,1.533827


### Ранжирование по скорости

Линейная интерполяция — самая быстрая: бинарный поиск интервала + одно деление. Кубический сплайн и B-сплайн требуют построения системы уравнений, что даёт линейную сложность по числу узлов. Motion-aware близок к сплайнам. Лагранж — самый медленный из-за пересчёта полиномиальных весов для каждой точки запроса в скользящем окне.

## Финальная таблица 3 ВКР

Объединяем данные о точности (из ноутбука 03) и производительности в одну таблицу.

In [5]:
df_accuracy = pd.read_csv(TABLES_DIR / "table_03_accuracy.csv")

df_time = df_bench[["Метод", "мкс/точка"]].rename(columns={"мкс/точка": "Время, мкс/точка"})

df_full = df_accuracy.merge(df_time, on="Метод")

df_full.to_csv(TABLES_DIR / "table_03_full.csv", index=False)
print(f"Сохранено: {TABLES_DIR / 'table_03_full.csv'}\n")
df_full.style.format({
    "RMSE, м": "{:.2f}",
    "Стандартное отклонение": "{:.2f}",
    "Время, мкс/точка": "{:.1f}",
}).hide(axis="index")

Сохранено: ../results/tables/table_03_full.csv



Метод,"RMSE, м",Стандартное отклонение,"Время, мкс/точка"
Линейная,9.01,5.89,0.2
Лагранж,2744.49,6530.54,59.0
Кубический сплайн,9.51,9.06,28.9
B-сплайн,9.54,9.07,2.7
Motion-aware,7.23,4.62,1.5


## Итог

Полученные значения времени соответствуют ожидаемой вычислительной сложности: $O(\log n)$ для линейной (бинарный поиск), $O(n)$ для кубического сплайна и B-сплайна (решение трёхдиагональной системы), $O(k^2 \cdot m)$ для Лагранжа из-за пересчёта весов в скользящем окне степени $k$ для каждой из $m$ точек запроса.